# Extra collocation points (50D, data-scarce)

**Paper:** *Gaeta–Lie Neural SDEs: Symmetry-Regularized Learning of Stochastic Dynamics* (NeurIPS 2026) — Appendix N.

Sweeps the number of extra Stage-III collocation points over {0, 128, 256, 512, 1024} on the 50D systems (near-trajectory Gaussian jitter).

**Pipeline naming used in the code:** Stage I = baseline neural SDE surrogate; Stage II = symmetry discovery over the basis library; Stage F = Stage III (symmetry-regularized refinement).

Run the cells top to bottom. All configuration lives in `CONFIG` in the first code cell. See the [repository README](../../README.md) for installation and the full experiment index.

## Cell 1 — E9 configuration, imports, reproducibility, and utilities

In [ ]:
# ============================================================
# Cell 1 — E9 configuration, imports, reproducibility, and utilities
# ============================================================

# These environment variables must be set before importing JAX.
import os
os.environ.setdefault("XLA_PYTHON_CLIENT_PREALLOCATE", "false")
os.environ.setdefault("XLA_PYTHON_CLIENT_MEM_FRACTION", "0.95")

from jax import config as jax_config
jax_config.update("jax_enable_x64", False)

import gc
import hashlib
import json
import math
import pickle
import subprocess
import threading
import time
from pathlib import Path

import jax
import jax.numpy as jnp
import numpy as np

print("JAX version:", jax.__version__)
print("JAX backend:", jax.default_backend())
print("JAX devices:", jax.devices())


CONFIG = {
    # Fixed E9 protocol.
    "STATE_DIM": 50,
    "N_TRAJECTORIES": 60,            # data-scarce regime: d + 10
    "TRAIN_FRACTION": 0.80,
    "E9_SYSTEMS": [
        "ring_double_well_50d",
        "low_rank_ou_50d",
        "tanh_chain_50d",
    ],
    "E9_EXTRA_POINT_BUDGETS": [0, 128, 256, 512, 1024],
    "E9_SEEDS": [0, 1, 2, 3, 4],
    "E9_JITTER_STD": 0.05,
    "E9_OUTPUT_DIR": "./e9_outputs",
    "E9_RESUME": True,
    "E9_CODE_VERSION": "e9_collocation_budget_v2_2026_07_28",

    # Neural SDE surrogate.
    "MLP_HIDDEN_DIM": 192,
    "MLP_DEPTH": 3,
    "SIGMA_EPS": 1e-4,

    # Stage I.
    "STAGE1_EPOCHS": 2000,
    "STAGE1_BATCH_SIZE": 512,
    "STAGE1_LR": 2e-3,

    # Stage II.
    "N_GENERATORS": 50,
    "STAGE2_EPOCHS": 4000,
    "STAGE2_LR": 5e-3,
    "STAGE2_POINTS": 256,
    "SYM_INIT_SCALE": 0.03,
    "LAMBDA_DET": 1.0,
    "LAMBDA_CLOSE": 0.5,
    "LAMBDA_IND": 0.05,
    "LAMBDA_PROJECTABLE": 1.0,
    "LAMBDA_NORM_HINGE": 0.01,
    "MIN_GENERATOR_NORM": 0.10,
    "CLOSURE_RCOND": 1e-5,

    # Editable high-dimensional basis.
    "BASIS_LINEAR_MU_LIST": [-1.0, -0.5, 0.5, 1.0],
    "BASIS_EXP_CLIP": 6.0,
    "BASIS_LOW_RANK_K": 6,
    "BASIS_ADD_EXPONENTIAL": True,
    "BASIS_ADD_LOW_RANK_ROTATIONS": True,
    "BASIS_ADD_LOW_RANK_SHEARS": True,

    # Stage III.
    "STAGE3_UPDATES": 800,
    "STAGE3_BATCH_SIZE": 512,
    "STAGE3_LR": 3e-4,
    "STAGE3_LAMBDA_SYM_DET": 105000.0,
    "STAGE3_LAMBDA_PROJECTABLE": 1e-3,
    "STAGE3_N_DATA_SYM_POINTS": 128,

    # Chunking makes the 1024-extra-point condition memory-safe while
    # preserving the exact mean determining residual.
    "DET_CHUNK_SIZE": 128,

    # Fixed coefficient-evaluation cloud for every budget in a system/seed pair.
    "N_EVAL_POINTS": 2000,

    # Logging and memory polling.
    "PRINT_EVERY_STAGE1": 250,
    "PRINT_EVERY_STAGE2": 500,
    "PRINT_EVERY_STAGE3": 200,
    "MEMORY_POLL_INTERVAL_S": 0.10,
}

D = int(CONFIG["STATE_DIM"])
Q = D + 1
E9_SYSTEMS = list(CONFIG["E9_SYSTEMS"])
E9_EXTRA_POINT_BUDGETS = list(CONFIG["E9_EXTRA_POINT_BUDGETS"])
E9_SEEDS = list(CONFIG["E9_SEEDS"])
E9_OUTPUT_DIR = CONFIG["E9_OUTPUT_DIR"]

OUT_DIR = Path(E9_OUTPUT_DIR)
CHECKPOINT_DIR = OUT_DIR / "checkpoints"
OUT_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)


def block_until_ready(tree):
    """Synchronize every JAX array in a pytree."""
    leaves = jax.tree_util.tree_leaves(tree)
    for leaf in leaves:
        if hasattr(leaf, "block_until_ready"):
            leaf.block_until_ready()
    return tree


def tree_to_host(tree):
    return jax.tree_util.tree_map(lambda x: np.asarray(jax.device_get(x)), tree)


def tree_to_device(tree):
    return jax.tree_util.tree_map(lambda x: jnp.asarray(x), tree)


def atomic_pickle_dump(obj, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + ".tmp")
    with open(tmp, "wb") as f:
        pickle.dump(obj, f, protocol=pickle.HIGHEST_PROTOCOL)
    os.replace(tmp, path)


def pickle_load(path):
    with open(path, "rb") as f:
        return pickle.load(f)


def stable_run_signature():
    payload = {
        "code_version": CONFIG["E9_CODE_VERSION"],
        "systems": CONFIG["E9_SYSTEMS"],
        "budgets": CONFIG["E9_EXTRA_POINT_BUDGETS"],
        "seeds": CONFIG["E9_SEEDS"],
        "n_trajectories": CONFIG["N_TRAJECTORIES"],
        "train_fraction": CONFIG["TRAIN_FRACTION"],
        "model": [CONFIG["MLP_HIDDEN_DIM"], CONFIG["MLP_DEPTH"], CONFIG["SIGMA_EPS"]],
        "stage1": [CONFIG["STAGE1_EPOCHS"], CONFIG["STAGE1_BATCH_SIZE"], CONFIG["STAGE1_LR"]],
        "stage2": [
            CONFIG["N_GENERATORS"], CONFIG["STAGE2_EPOCHS"], CONFIG["STAGE2_LR"],
            CONFIG["STAGE2_POINTS"], CONFIG["E9_JITTER_STD"],
            CONFIG["LAMBDA_DET"], CONFIG["LAMBDA_CLOSE"], CONFIG["LAMBDA_IND"],
            CONFIG["LAMBDA_PROJECTABLE"], CONFIG["LAMBDA_NORM_HINGE"],
        ],
        "basis": [
            CONFIG["BASIS_LINEAR_MU_LIST"], CONFIG["BASIS_LOW_RANK_K"],
            CONFIG["BASIS_ADD_EXPONENTIAL"],
            CONFIG["BASIS_ADD_LOW_RANK_ROTATIONS"],
            CONFIG["BASIS_ADD_LOW_RANK_SHEARS"],
        ],
        "stage3": [
            CONFIG["STAGE3_UPDATES"], CONFIG["STAGE3_BATCH_SIZE"], CONFIG["STAGE3_LR"],
            CONFIG["STAGE3_LAMBDA_SYM_DET"], CONFIG["STAGE3_LAMBDA_PROJECTABLE"],
            CONFIG["STAGE3_N_DATA_SYM_POINTS"], CONFIG["DET_CHUNK_SIZE"],
        ],
        "n_eval": CONFIG["N_EVAL_POINTS"],
    }
    text = json.dumps(payload, sort_keys=True, separators=(",", ":"))
    return hashlib.sha256(text.encode("utf-8")).hexdigest()[:20]


E9_RUN_SIGNATURE = stable_run_signature()
print("E9 run signature:", E9_RUN_SIGNATURE)


def _gpu_memory_for_current_pid_mb():
    """Return current process GPU memory from nvidia-smi, or None."""
    if jax.default_backend() != "gpu":
        return None
    try:
        cmd = [
            "nvidia-smi",
            "--query-compute-apps=pid,used_memory",
            "--format=csv,noheader,nounits",
        ]
        output = subprocess.check_output(cmd, stderr=subprocess.DEVNULL, text=True)
        pid = os.getpid()
        total = 0.0
        found = False
        for line in output.strip().splitlines():
            parts = [p.strip() for p in line.split(",")]
            if len(parts) != 2:
                continue
            if int(parts[0]) == pid:
                total += float(parts[1])
                found = True
        return total if found else 0.0
    except Exception:
        return None


def _cpu_rss_mb():
    try:
        import psutil
        return psutil.Process(os.getpid()).memory_info().rss / (1024.0 ** 2)
    except Exception:
        return 0.0


def current_process_memory_mb():
    gpu = _gpu_memory_for_current_pid_mb()
    if gpu is not None:
        return float(gpu), "gpu_process_memory_nvidia_smi"
    return float(_cpu_rss_mb()), "cpu_rss_fallback"


class PeakMemoryMonitor:
    """Poll process memory during one complete Stage-III run."""

    def __init__(self, interval_s=0.10):
        self.interval_s = float(interval_s)
        self.stop_event = threading.Event()
        self.thread = None
        self.start_mb = 0.0
        self.peak_mb = 0.0
        self.source = "unknown"

    def _poll(self):
        while not self.stop_event.is_set():
            value, source = current_process_memory_mb()
            self.source = source
            self.peak_mb = max(self.peak_mb, value)
            self.stop_event.wait(self.interval_s)

    def start(self):
        self.start_mb, self.source = current_process_memory_mb()
        self.peak_mb = self.start_mb
        self.stop_event.clear()
        self.thread = threading.Thread(target=self._poll, daemon=True)
        self.thread.start()
        return self

    def stop(self):
        self.stop_event.set()
        if self.thread is not None:
            self.thread.join(timeout=max(1.0, 5.0 * self.interval_s))
        value, source = current_process_memory_mb()
        self.source = source
        self.peak_mb = max(self.peak_mb, value)
        return {
            "peak_mb": float(self.peak_mb),
            "delta_mb": float(max(0.0, self.peak_mb - self.start_mb)),
            "start_mb": float(self.start_mb),
            "source": self.source,
        }


## Cell 2 — The three manuscript 50D systems and deterministic data generation

In [ ]:
# ============================================================
# Cell 2 — The three manuscript 50D systems and deterministic data generation
# ============================================================


def _idx_float():
    return jnp.arange(D, dtype=jnp.float32)


def _phase_vector():
    return 2.0 * jnp.pi * _idx_float() / float(D)


def _mode_sin(k):
    return jnp.sin(2.0 * jnp.pi * k * _idx_float() / float(D))


def _mode_cos(k):
    return jnp.cos(2.0 * jnp.pi * k * _idx_float() / float(D))


def _normalize_mode(v):
    return v / (jnp.linalg.norm(v) + 1e-8)


def drift_ring_double_well_50d(t, x):
    c = 0.25
    lap = jnp.roll(x, 1, axis=-1) + jnp.roll(x, -1, axis=-1) - 2.0 * x
    forcing = 0.08 * jnp.sin(2.0 * jnp.pi * t + _phase_vector())
    return x - x**3 + c * lap + forcing


def sigma_ring_double_well_50d(t, x):
    phase = _phase_vector()
    base = 0.18 + 0.02 * jnp.sin(phase) ** 2
    return base + 0.04 * jax.nn.sigmoid(x)


def drift_low_rank_ou_50d(t, x):
    idx = _idx_float()
    lam = 0.45 + 0.30 * idx / float(D - 1)
    skew = 0.18 * (jnp.roll(x, -1, axis=-1) - jnp.roll(x, 1, axis=-1))

    v1 = _normalize_mode(_mode_sin(1))
    v2 = _normalize_mode(_mode_cos(2))
    v3 = _normalize_mode(_mode_sin(3))
    p1 = jnp.sum(x * v1, axis=-1, keepdims=True)
    p2 = jnp.sum(x * v2, axis=-1, keepdims=True)
    p3 = jnp.sum(x * v3, axis=-1, keepdims=True)

    low_rank = 0.45 * p1 * v2 - 0.35 * p2 * v1 + 0.25 * jnp.tanh(p3) * v3
    forcing = 0.05 * (jnp.sin(1.2 * t) * v1 + jnp.cos(0.8 * t) * v2)
    return -lam * x + skew + low_rank + forcing + 0.05 * jnp.tanh(x)


def sigma_low_rank_ou_50d(t, x):
    v1 = _normalize_mode(_mode_sin(1))
    p1 = jnp.sum(x * v1, axis=-1, keepdims=True)
    base = 0.20 + 0.03 * jnp.sin(_phase_vector()) ** 2
    return base + 0.03 * jax.nn.sigmoid(p1) + 0.02 * jnp.tanh(x) ** 2


def drift_tanh_chain_50d(t, x):
    phase = _phase_vector()
    local = (
        -0.55 * x
        + 0.35 * jnp.tanh(jnp.roll(x, -1, axis=-1))
        - 0.25 * jnp.tanh(jnp.roll(x, 1, axis=-1))
        + 0.10 * jnp.tanh(jnp.roll(x, -2, axis=-1))
    )
    v1 = _normalize_mode(_mode_cos(1))
    v2 = _normalize_mode(_mode_sin(2))
    p1 = jnp.sum(x * v1, axis=-1, keepdims=True)
    p2 = jnp.sum(x * v2, axis=-1, keepdims=True)
    global_feedback = 0.20 * jnp.tanh(p1) * v2 - 0.15 * jnp.tanh(p2) * v1
    forcing = 0.04 * jnp.sin(1.5 * t + phase)
    return local + global_feedback + forcing


def sigma_tanh_chain_50d(t, x):
    return 0.22 + 0.04 * jnp.sin(x) ** 2


SDE_REGISTRY = {
    "ring_double_well_50d": {
        "description": "Ex 50.1: ring-coupled double-well",
        "drift_fn": drift_ring_double_well_50d,
        "sigma_fn": sigma_ring_double_well_50d,
        "t_final": 2.0,
        "dt": 0.01,
        "x0_mean": 0.0,
        "x0_std": 0.75,
        "input_x_scale": 2.5,
        "x_box": (-2.5, 2.5),
    },
    "low_rank_ou_50d": {
        "description": "Ex 50.2: low-rank coupled OU/spiral",
        "drift_fn": drift_low_rank_ou_50d,
        "sigma_fn": sigma_low_rank_ou_50d,
        "t_final": 2.0,
        "dt": 0.01,
        "x0_mean": 0.0,
        "x0_std": 1.0,
        "input_x_scale": 3.0,
        "x_box": (-3.0, 3.0),
    },
    "tanh_chain_50d": {
        "description": "Ex 50.3: tanh reaction chain with low-rank feedback",
        "drift_fn": drift_tanh_chain_50d,
        "sigma_fn": sigma_tanh_chain_50d,
        "t_final": 3.0,
        "dt": 0.01,
        "x0_mean": 0.0,
        "x0_std": 1.0,
        "input_x_scale": 3.0,
        "x_box": (-3.0, 3.0),
    },
}


def make_problem(sde_name):
    if sde_name not in SDE_REGISTRY:
        raise ValueError(f"Unknown SDE {sde_name}; choices are {list(SDE_REGISTRY)}")
    spec = dict(SDE_REGISTRY[sde_name])
    spec["name"] = sde_name
    spec["input_t_scale"] = float(spec["t_final"])
    spec["t_box"] = (0.0, float(spec["t_final"]))
    return spec


def _as_state_vector(value, name):
    arr = jnp.asarray(value, dtype=jnp.float32)
    if arr.ndim == 0:
        return jnp.full((D,), arr, dtype=jnp.float32)
    if arr.shape != (D,):
        raise ValueError(f"{name} must be scalar or shape ({D},), got {arr.shape}")
    return arr


def build_data_functions(problem):
    true_drift = problem["drift_fn"]
    true_sigma_raw = problem["sigma_fn"]
    dt = float(problem["dt"])
    n_steps = int(round(float(problem["t_final"]) / dt))
    n_paths = int(CONFIG["N_TRAJECTORIES"])
    x0_mean = _as_state_vector(problem["x0_mean"], "x0_mean")
    x0_std = _as_state_vector(problem["x0_std"], "x0_std")

    def true_sigma(t, x):
        return jnp.maximum(true_sigma_raw(t, x), 1e-6)

    @jax.jit
    def simulate_paths(key):
        key_x0, key_noise = jax.random.split(key)
        x0 = x0_mean[None, :] + x0_std[None, :] * jax.random.normal(
            key_x0, (n_paths, D), dtype=jnp.float32
        )
        noise = jax.random.normal(
            key_noise, (n_steps, n_paths, D), dtype=jnp.float32
        )
        times = jnp.arange(n_steps + 1, dtype=jnp.float32) * dt
        sqrt_dt = jnp.sqrt(jnp.asarray(dt, dtype=jnp.float32))

        def step(x, inputs):
            t, zeta = inputs
            x_next = x + true_drift(t, x) * dt + true_sigma(t, x) * sqrt_dt * zeta
            return x_next, x_next

        _, tail = jax.lax.scan(step, x0, (times[:-1], noise))
        paths = jnp.concatenate([x0[None, :, :], tail], axis=0)
        return times, jnp.transpose(paths, (1, 0, 2))

    return true_drift, true_sigma, simulate_paths


def make_increment_dataset(times, paths, dt):
    n_paths, n_times, dim = paths.shape
    n_steps = n_times - 1
    t0 = jnp.broadcast_to(times[:-1][None, :], (n_paths, n_steps)).reshape(-1)
    x0 = paths[:, :-1, :].reshape(-1, dim)
    dx = (paths[:, 1:, :] - paths[:, :-1, :]).reshape(-1, dim)
    dt_arr = jnp.full((x0.shape[0],), float(dt), dtype=jnp.float32)
    return t0, x0, dt_arr, dx


def split_increment_dataset(key, dataset, train_fraction):
    n = int(dataset[0].shape[0])
    perm = jax.random.permutation(key, n)
    n_train = int(float(train_fraction) * n)
    train_idx, test_idx = perm[:n_train], perm[n_train:]
    train = tuple(arr[train_idx] for arr in dataset)
    test = tuple(arr[test_idx] for arr in dataset)
    return train, test


def sample_batch(key, dataset, batch_size):
    n = dataset[0].shape[0]
    idx = jax.random.randint(key, (int(batch_size),), 0, n)
    return tuple(arr[idx] for arr in dataset)


def sample_tx_data_jitter(key, dataset, n_points, jitter_std):
    n_points = int(n_points)
    if n_points == 0:
        return jnp.zeros((0, Q), dtype=jnp.float32)
    key_idx, key_noise = jax.random.split(key)
    t0, x0, _, _ = dataset
    idx = jax.random.randint(key_idx, (n_points,), 0, t0.shape[0])
    t = t0[idx]
    x = x0[idx]
    noise = float(jitter_std) * jax.random.normal(key_noise, x.shape, dtype=x.dtype)
    return jnp.concatenate([t[:, None], x + noise], axis=1)


def _x_box_arrays(x_box):
    low, high = x_box
    low = jnp.asarray(low, dtype=jnp.float32)
    high = jnp.asarray(high, dtype=jnp.float32)
    if low.ndim == 0:
        low = jnp.full((D,), low, dtype=jnp.float32)
    if high.ndim == 0:
        high = jnp.full((D,), high, dtype=jnp.float32)
    return low, high


def sample_tx_box(key, n_points, t_box, x_box):
    key_t, key_x = jax.random.split(key)
    t = jax.random.uniform(
        key_t, (int(n_points), 1), minval=float(t_box[0]), maxval=float(t_box[1])
    )
    low, high = _x_box_arrays(x_box)
    u = jax.random.uniform(key_x, (int(n_points), D), dtype=jnp.float32)
    x = low[None, :] + u * (high[None, :] - low[None, :])
    return jnp.concatenate([t, x], axis=1)


def generate_data_bundle(problem, seed, system_index):
    true_drift, true_sigma, simulate_paths = build_data_functions(problem)
    root = jax.random.fold_in(jax.random.PRNGKey(int(seed)), int(system_index))
    key_sim = jax.random.fold_in(root, 11)
    key_split = jax.random.fold_in(root, 12)
    key_eval = jax.random.fold_in(root, 13)

    times, paths = simulate_paths(key_sim)
    block_until_ready(paths)
    dataset = make_increment_dataset(times, paths, problem["dt"])
    train_data, test_data = split_increment_dataset(
        key_split, dataset, CONFIG["TRAIN_FRACTION"]
    )
    eval_tx = sample_tx_box(
        key_eval, CONFIG["N_EVAL_POINTS"], problem["t_box"], problem["x_box"]
    )
    eval_t, eval_x = eval_tx[:, 0], eval_tx[:, 1:]
    eval_true_f = jax.vmap(true_drift)(eval_t, eval_x)
    eval_true_sigma = jax.vmap(true_sigma)(eval_t, eval_x)
    block_until_ready((eval_true_f, eval_true_sigma))

    return {
        "root_key": root,
        "times": times,
        "paths": paths,
        "train_data": train_data,
        "test_data": test_data,
        "eval_tx": eval_tx,
        "eval_true_f": eval_true_f,
        "eval_true_sigma": eval_true_sigma,
        "true_drift": true_drift,
        "true_sigma": true_sigma,
    }

## Cell 3 — Surrogate network, optimizer, and Stage-I primitives

In [ ]:
# ============================================================
# Cell 3 — Surrogate network, optimizer, and Stage-I primitives
# ============================================================


def glorot_uniform(key, in_dim, out_dim):
    limit = jnp.sqrt(6.0 / float(in_dim + out_dim))
    return jax.random.uniform(
        key, (in_dim, out_dim), minval=-limit, maxval=limit, dtype=jnp.float32
    )


class MLP:
    def __init__(self, in_dim, hidden_dim, depth, out_dim):
        self.in_dim = int(in_dim)
        self.hidden_dim = int(hidden_dim)
        self.depth = int(depth)
        self.out_dim = int(out_dim)

    def init(self, key):
        sizes = [self.in_dim] + [self.hidden_dim] * self.depth + [self.out_dim]
        keys = jax.random.split(key, len(sizes) - 1)
        params = []
        for k, in_dim, out_dim in zip(keys, sizes[:-1], sizes[1:]):
            params.append({
                "W": glorot_uniform(k, in_dim, out_dim),
                "b": jnp.zeros((out_dim,), dtype=jnp.float32),
            })
        return params

    def apply(self, params, z):
        h = z
        for layer in params[:-1]:
            h = jnp.tanh(h @ layer["W"] + layer["b"])
        return h @ params[-1]["W"] + params[-1]["b"]


def adam_init(params):
    return {
        "m": jax.tree_util.tree_map(jnp.zeros_like, params),
        "v": jax.tree_util.tree_map(jnp.zeros_like, params),
        "t": jnp.asarray(0, dtype=jnp.int32),
    }


def adam_update(params, grads, state, lr, beta1=0.9, beta2=0.999, eps=1e-8):
    t = state["t"] + 1
    m = jax.tree_util.tree_map(
        lambda old, g: beta1 * old + (1.0 - beta1) * g, state["m"], grads
    )
    v = jax.tree_util.tree_map(
        lambda old, g: beta2 * old + (1.0 - beta2) * g * g, state["v"], grads
    )
    m_hat = jax.tree_util.tree_map(lambda x: x / (1.0 - beta1**t), m)
    v_hat = jax.tree_util.tree_map(lambda x: x / (1.0 - beta2**t), v)
    new_params = jax.tree_util.tree_map(
        lambda p, mm, vv: p - float(lr) * mm / (jnp.sqrt(vv) + eps),
        params, m_hat, v_hat,
    )
    return new_params, {"m": m, "v": v, "t": t}


def build_surrogate_functions(problem):
    model = MLP(
        in_dim=Q,
        hidden_dim=CONFIG["MLP_HIDDEN_DIM"],
        depth=CONFIG["MLP_DEPTH"],
        out_dim=2 * D,
    )
    t_scale = float(problem["input_t_scale"])
    x_scale = float(problem["input_x_scale"])
    sigma_eps = float(CONFIG["SIGMA_EPS"])

    def surrogate_f_sigma(params, t, x):
        z = jnp.concatenate([
            jnp.asarray([t / t_scale], dtype=jnp.float32),
            x / x_scale,
        ])
        raw = model.apply(params, z)
        return raw[:D], jax.nn.softplus(raw[D:]) + sigma_eps

    def f_vec_single(params, z):
        return surrogate_f_sigma(params, z[0], z[1:])[0]

    def sigma_vec_single(params, z):
        return surrogate_f_sigma(params, z[0], z[1:])[1]

    def data_nll_loss(params, batch):
        t0, x0, dt, dx = batch
        f, sigma = jax.vmap(lambda tt, xx: surrogate_f_sigma(params, tt, xx))(t0, x0)
        mean = f * dt[:, None]
        var = sigma**2 * dt[:, None] + 1e-8
        nll = 0.5 * ((dx - mean) ** 2 / var + jnp.log(2.0 * jnp.pi * var))
        return jnp.mean(jnp.sum(nll, axis=-1))

    @jax.jit
    def stage1_step(params, opt_state, batch):
        loss, grads = jax.value_and_grad(data_nll_loss)(params, batch)
        params, opt_state = adam_update(
            params, grads, opt_state, CONFIG["STAGE1_LR"]
        )
        return params, opt_state, loss

    @jax.jit
    def predict_batch(params, t, x):
        return jax.vmap(lambda tt, xx: surrogate_f_sigma(params, tt, xx))(t, x)

    return {
        "model": model,
        "surrogate_f_sigma": surrogate_f_sigma,
        "f_vec_single": f_vec_single,
        "sigma_vec_single": sigma_vec_single,
        "data_nll_loss": data_nll_loss,
        "stage1_step": stage1_step,
        "predict_batch": predict_batch,
    }


def train_stage1(ops, train_data, test_data, root_key, label):
    key_init = jax.random.fold_in(root_key, 101)
    key_batches = jax.random.fold_in(root_key, 102)
    params = ops["model"].init(key_init)
    opt_state = adam_init(params)

    t0 = time.perf_counter()
    last_loss = None
    for step in range(1, CONFIG["STAGE1_EPOCHS"] + 1):
        batch_key = jax.random.fold_in(key_batches, step)
        batch = sample_batch(batch_key, train_data, CONFIG["STAGE1_BATCH_SIZE"])
        params, opt_state, last_loss = ops["stage1_step"](params, opt_state, batch)

        if step == 1 or step % CONFIG["PRINT_EVERY_STAGE1"] == 0:
            batch_loss = float(last_loss)
            print(f"[{label} | Stage I] step {step:4d}/{CONFIG['STAGE1_EPOCHS']} | batch NLL {batch_loss:.6e}")

    block_until_ready(last_loss)
    elapsed = time.perf_counter() - t0
    train_nll = float(ops["data_nll_loss"](params, train_data))
    test_nll = float(ops["data_nll_loss"](params, test_data))
    print(f"[{label} | Stage I] done in {elapsed:.2f}s | train NLL {train_nll:.6e} | test NLL {test_nll:.6e}")
    return params, elapsed


## Cell 4 — Per-system editable Gaeta/low-rank basis factory

In [ ]:
# ============================================================
# Cell 4 — Per-system editable Gaeta/low-rank basis factory
# ============================================================


def build_basis_library(problem):
    library = []
    t_scale = float(problem["input_t_scale"])

    def add(name, source, fn):
        library.append({"name": name, "source": source, "fn": fn})

    def time_basis(t):
        return t / t_scale

    def safe_exp(value):
        clip = float(CONFIG["BASIS_EXP_CLIP"])
        return jnp.exp(jnp.clip(value, -clip, clip))

    def zero_xi():
        return jnp.zeros((D,), dtype=jnp.float32)

    def e_i(i):
        return jnp.zeros((D,), dtype=jnp.float32).at[i].set(1.0)

    def unit(v):
        return v / (jnp.linalg.norm(v) + 1e-8)

    def low_rank_directions(k_count):
        idx = jnp.arange(D, dtype=jnp.float32)
        directions = [unit(jnp.ones((D,), dtype=jnp.float32))]
        freq = 1
        while len(directions) < int(k_count):
            phase = 2.0 * jnp.pi * freq * (idx + 0.5) / float(D)
            directions.append(unit(jnp.sin(phase)))
            if len(directions) < int(k_count):
                directions.append(unit(jnp.cos(phase)))
            freq += 1
        return tuple(directions[:int(k_count)])

    # Time-only templates.
    add("time_translation__dt", "time", lambda t, x: jnp.concatenate([
        jnp.asarray([1.0], dtype=jnp.float32), zero_xi()
    ]))
    add("time_scaling__t_dt", "time", lambda t, x: jnp.concatenate([
        jnp.asarray([time_basis(t)], dtype=jnp.float32), zero_xi()
    ]))
    add("time_quadratic__t2_dt", "time", lambda t, x: jnp.concatenate([
        jnp.asarray([time_basis(t) ** 2], dtype=jnp.float32), zero_xi()
    ]))

    # Coordinate translations and scalings.
    for i in range(D):
        add(
            f"translation__dx{i+1}", "translation",
            lambda t, x, i=i: jnp.concatenate([
                jnp.asarray([0.0], dtype=jnp.float32), e_i(i)
            ]),
        )
    for i in range(D):
        add(
            f"scaling__x{i+1}_dx{i+1}", "scaling",
            lambda t, x, i=i: jnp.concatenate([
                jnp.asarray([0.0], dtype=jnp.float32),
                jnp.zeros((D,), dtype=jnp.float32).at[i].set(x[i]),
            ]),
        )

    # Lifted Brownian/Gaeta scaling.
    add(
        "global_brownian_scaling__2t_dt_plus_x_dot_dx", "global_scaling",
        lambda t, x: jnp.concatenate([
            jnp.asarray([2.0 * time_basis(t)], dtype=jnp.float32), x
        ]),
    )

    # Exponential Gaeta templates.
    if CONFIG["BASIS_ADD_EXPONENTIAL"]:
        for mu_value in CONFIG["BASIS_LINEAR_MU_LIST"]:
            mu = float(mu_value)
            if abs(mu) < 1e-12:
                continue
            for i in range(D):
                add(
                    f"exp_translation__mu_{mu:+.2f}__dx{i+1}", "exp_translation",
                    lambda t, x, mu=mu, i=i: jnp.concatenate([
                        jnp.asarray([0.0], dtype=jnp.float32),
                        jnp.zeros((D,), dtype=jnp.float32).at[i].set(
                            safe_exp(mu * time_basis(t))
                        ),
                    ]),
                )
            add(
                f"exp_projective__mu_{mu:+.2f}", "exp_projective",
                lambda t, x, mu=mu: jnp.concatenate([
                    jnp.asarray([safe_exp(2.0 * mu * time_basis(t))], dtype=jnp.float32),
                    mu * safe_exp(2.0 * mu * time_basis(t)) * x,
                ]),
            )

    directions = low_rank_directions(CONFIG["BASIS_LOW_RANK_K"])
    k_low = len(directions)

    if CONFIG["BASIS_ADD_LOW_RANK_ROTATIONS"]:
        for a in range(k_low):
            for b in range(a + 1, k_low):
                va, vb = directions[a], directions[b]
                add(
                    f"lowrank_rotation__v{a+1}_v{b+1}", "lowrank_rotation",
                    lambda t, x, va=va, vb=vb: jnp.concatenate([
                        jnp.asarray([0.0], dtype=jnp.float32),
                        jnp.dot(vb, x) * va - jnp.dot(va, x) * vb,
                    ]),
                )

    if CONFIG["BASIS_ADD_LOW_RANK_SHEARS"]:
        for a in range(k_low - 1):
            va, vb = directions[a], directions[a + 1]
            add(
                f"lowrank_shear__v{a+2}_to_v{a+1}", "lowrank_shear",
                lambda t, x, va=va, vb=vb: jnp.concatenate([
                    jnp.asarray([0.0], dtype=jnp.float32), jnp.dot(vb, x) * va
                ]),
            )
            add(
                f"lowrank_shear__v{a+1}_to_v{a+2}", "lowrank_shear",
                lambda t, x, va=va, vb=vb: jnp.concatenate([
                    jnp.asarray([0.0], dtype=jnp.float32), jnp.dot(va, x) * vb
                ]),
            )

    names = [item["name"] for item in library]
    sources = [item["source"] for item in library]
    functions = tuple(item["fn"] for item in library)

    def eval_basis_one(t, x):
        return jnp.stack([fn(t, x) for fn in functions], axis=0)

    def eval_generators_point(W, z):
        return W @ eval_basis_one(z[0], z[1:])

    def eval_generators_batch(W, tx):
        return jax.vmap(lambda z: eval_generators_point(W, z))(tx)

    counts = {source: sources.count(source) for source in sorted(set(sources))}
    return {
        "names": names,
        "sources": sources,
        "functions": functions,
        "n_basis": len(functions),
        "counts": counts,
        "eval_basis_one": eval_basis_one,
        "eval_generators_point": eval_generators_point,
        "eval_generators_batch": eval_generators_batch,
    }

## Cell 5 — Determining equations, closure, Stage II, and complete per-system ops

In [ ]:
# ============================================================
# Cell 5 — Determining equations, closure, Stage II, and complete per-system ops
# ============================================================


def build_symmetry_and_training_ops(problem):
    surrogate = build_surrogate_functions(problem)
    basis = build_basis_library(problem)
    f_vec_single = surrogate["f_vec_single"]
    sigma_vec_single = surrogate["sigma_vec_single"]
    eval_generators_point = basis["eval_generators_point"]
    eval_generators_batch = basis["eval_generators_batch"]
    eye_d = jnp.eye(D, dtype=jnp.float32)
    chunk_size = int(CONFIG["DET_CHUNK_SIZE"])

    def det_residuals_point(params, W, z):
        f = f_vec_single(params, z)
        sigma = sigma_vec_single(params, z)
        jac_f = jax.jacfwd(lambda zz: f_vec_single(params, zz))(z)
        jac_sigma = jax.jacfwd(lambda zz: sigma_vec_single(params, zz))(z)

        f_t, f_x = jac_f[:, 0], jac_f[:, 1:]
        sigma_t, sigma_x = jac_sigma[:, 0], jac_sigma[:, 1:]

        gen_fn = lambda zz: eval_generators_point(W, zz)
        generators = gen_fn(z)
        jac_gen = jax.jacfwd(gen_fn)(z)

        tau = generators[:, 0]
        xi = generators[:, 1:]
        tau_t = jac_gen[:, 0, 0]
        tau_x = jac_gen[:, 0, 1:]
        xi_t = jac_gen[:, 1:, 0]
        xi_x = jac_gen[:, 1:, 1:]

        # Every library xi is at most linear in x, so the Ito Hessian term is zero.
        adv_xi = jnp.einsum("j,aij->ai", f, xi_x)
        xi_grad_f = jnp.einsum("aj,ij->ai", xi, f_x)
        r_f = xi_t + adv_xi - xi_grad_f - tau[:, None] * f_t[None, :] - tau_t[:, None] * f[None, :]

        first_sigma = jnp.einsum("p,aip->aip", sigma, xi_x)
        diagonal_correction = (
            jnp.einsum("aj,ij->ai", xi, sigma_x)
            + tau[:, None] * sigma_t[None, :]
            + 0.5 * tau_t[:, None] * sigma[None, :]
        )
        r_sigma = first_sigma - diagonal_correction[:, :, None] * eye_d[None, :, :]
        return r_f, r_sigma, tau_x

    def chunk_sums(params, W, tx_chunk, mask_chunk):
        r_f, r_sigma, tau_x = jax.vmap(
            lambda z: det_residuals_point(params, W, z)
        )(tx_chunk)
        mask_f = mask_chunk[:, None, None]
        mask_sigma = mask_chunk[:, None, None, None]
        return jnp.stack([
            jnp.sum((r_f**2) * mask_f),
            jnp.sum((r_sigma**2) * mask_sigma),
            jnp.sum((tau_x**2) * mask_f),
        ])

    chunk_sums_remat = jax.checkpoint(chunk_sums)

    def determining_loss(params, W, tx):
        n_points = int(tx.shape[0])
        if n_points <= 0:
            raise ValueError("determining_loss requires at least one collocation point")
        pad = (-n_points) % chunk_size
        tx_padded = jnp.pad(tx, ((0, pad), (0, 0)))
        mask = jnp.concatenate([
            jnp.ones((n_points,), dtype=jnp.float32),
            jnp.zeros((pad,), dtype=jnp.float32),
        ])
        tx_chunks = tx_padded.reshape((-1, chunk_size, Q))
        mask_chunks = mask.reshape((-1, chunk_size))

        def body(total, inputs):
            tx_chunk, mask_chunk = inputs
            return total + chunk_sums_remat(params, W, tx_chunk, mask_chunk), None

        totals, _ = jax.lax.scan(
            body, jnp.zeros((3,), dtype=jnp.float32), (tx_chunks, mask_chunks)
        )
        m = int(W.shape[0])
        drift_mean = totals[0] / float(n_points * m * D)
        sigma_mean = totals[1] / float(n_points * m * D * D)
        projectable_mean = totals[2] / float(n_points * m * D)
        return drift_mean + sigma_mean, projectable_mean

    def generator_jacobians_batch(W, tx):
        def one(z):
            fn = lambda zz: eval_generators_point(W, zz)
            value = fn(z)
            return value, jax.jacfwd(fn)(z)
        return jax.vmap(one)(tx)

    def lie_closure_loss_global(W, tx):
        m = int(W.shape[0])
        if m < 2:
            return jnp.asarray(0.0, dtype=jnp.float32)
        generators, jacobians = generator_jacobians_batch(W, tx)
        idx_i, idx_j = jnp.triu_indices(m, k=1)
        xi = generators[:, idx_i, :]
        xj = generators[:, idx_j, :]
        ji = jacobians[:, idx_i, :, :]
        jj = jacobians[:, idx_j, :, :]
        brackets = (
            jnp.einsum("spc,spqc->spq", xi, jj)
            - jnp.einsum("spc,spqc->spq", xj, ji)
        )
        design = jnp.transpose(generators, (1, 0, 2)).reshape(m, -1).T
        targets = jnp.transpose(brackets, (1, 0, 2)).reshape(idx_i.shape[0], -1).T
        gram = design.T @ design
        reg = float(CONFIG["CLOSURE_RCOND"]) ** 2 * jnp.eye(m, dtype=gram.dtype)
        constants = jnp.linalg.solve(gram + reg, design.T @ targets)
        return jnp.mean((targets - design @ constants) ** 2)

    def gram_independence_loss(W, tx):
        values = eval_generators_batch(W, tx)
        m = int(values.shape[1])
        design = jnp.transpose(values, (1, 0, 2)).reshape(m, -1).T
        gram = (design.T @ design) / float(values.shape[0] * Q)
        identity = jnp.eye(m, dtype=gram.dtype)
        independence = jnp.mean((gram - identity) ** 2)
        norm_hinge = jnp.mean(
            jnp.maximum(0.0, float(CONFIG["MIN_GENERATOR_NORM"]) - jnp.diag(gram)) ** 2
        )
        return independence, norm_hinge, gram

    def stage2_objective(W, frozen_params, tx):
        det, projectable = determining_loss(frozen_params, W, tx)
        closure = lie_closure_loss_global(W, tx)
        independence, norm_hinge, _ = gram_independence_loss(W, tx)
        total = (
            float(CONFIG["LAMBDA_DET"]) * det
            + float(CONFIG["LAMBDA_CLOSE"]) * closure
            + float(CONFIG["LAMBDA_IND"]) * independence
            + float(CONFIG["LAMBDA_PROJECTABLE"]) * projectable
            + float(CONFIG["LAMBDA_NORM_HINGE"]) * norm_hinge
        )
        return total, {
            "total": total,
            "det": det,
            "closure": closure,
            "independence": independence,
            "projectable": projectable,
            "norm_hinge": norm_hinge,
        }

    @jax.jit
    def stage2_step(W, opt_state, frozen_params, tx):
        (_, aux), grads = jax.value_and_grad(stage2_objective, has_aux=True)(
            W, frozen_params, tx
        )
        W, opt_state = adam_update(W, grads, opt_state, CONFIG["STAGE2_LR"])
        return W, opt_state, aux

    def stage3_objective(params, W_fixed, batch, tx_extra):
        data_loss = surrogate["data_nll_loss"](params, batch)
        t_batch, x_batch, _, _ = batch
        n_data_sym = min(int(CONFIG["STAGE3_N_DATA_SYM_POINTS"]), int(t_batch.shape[0]))
        tx_data = jnp.concatenate([
            t_batch[:n_data_sym, None], x_batch[:n_data_sym]
        ], axis=1)
        tx_sym = jnp.concatenate([tx_data, tx_extra], axis=0)
        det, projectable = determining_loss(params, W_fixed, tx_sym)
        total = (
            data_loss
            + float(CONFIG["STAGE3_LAMBDA_SYM_DET"]) * det
            + float(CONFIG["STAGE3_LAMBDA_PROJECTABLE"]) * projectable
        )
        return total, {
            "total": total,
            "data": data_loss,
            "sym_det": det,
            "projectable": projectable,
        }

    def stage3_step_impl(params, opt_state, W_fixed, batch, tx_extra):
        (_, aux), grads = jax.value_and_grad(stage3_objective, has_aux=True)(
            params, W_fixed, batch, tx_extra
        )
        params, opt_state = adam_update(params, grads, opt_state, CONFIG["STAGE3_LR"])
        return params, opt_state, aux

    return {
        **surrogate,
        "basis": basis,
        "determining_loss": determining_loss,
        "gram_independence_loss": gram_independence_loss,
        "stage2_step": stage2_step,
        "stage3_step_impl": stage3_step_impl,
    }


def train_stage2(ops, frozen_params, train_data, root_key, label):
    key_init = jax.random.fold_in(root_key, 201)
    key_points = jax.random.fold_in(root_key, 202)
    W = float(CONFIG["SYM_INIT_SCALE"]) * jax.random.normal(
        key_init,
        (int(CONFIG["N_GENERATORS"]), int(ops["basis"]["n_basis"])),
        dtype=jnp.float32,
    )
    opt_state = adam_init(W)

    t0 = time.perf_counter()
    last_aux = None
    for step in range(1, CONFIG["STAGE2_EPOCHS"] + 1):
        tx_key = jax.random.fold_in(key_points, step)
        tx = sample_tx_data_jitter(
            tx_key,
            train_data,
            CONFIG["STAGE2_POINTS"],
            CONFIG["E9_JITTER_STD"],
        )
        W, opt_state, last_aux = ops["stage2_step"](
            W, opt_state, frozen_params, tx
        )
        if step == 1 or step % CONFIG["PRINT_EVERY_STAGE2"] == 0:
            values = {k: float(v) for k, v in last_aux.items()}
            print(
                f"[{label} | Stage II] step {step:4d}/{CONFIG['STAGE2_EPOCHS']} | "
                f"total {values['total']:.4e} | det {values['det']:.4e} | "
                f"closure {values['closure']:.4e} | ind {values['independence']:.4e}"
            )

    block_until_ready(last_aux)
    elapsed = time.perf_counter() - t0
    eval_key = jax.random.fold_in(key_points, 999999)
    tx_eval = sample_tx_data_jitter(
        eval_key, train_data, CONFIG["STAGE2_POINTS"], CONFIG["E9_JITTER_STD"]
    )
    det, proj = ops["determining_loss"](frozen_params, W, tx_eval)
    _, _, gram = ops["gram_independence_loss"](W, tx_eval)
    print(
        f"[{label} | Stage II] done in {elapsed:.2f}s | held-out det {float(det):.4e} | "
        f"Gram diag [{float(jnp.min(jnp.diag(gram))):.3e}, {float(jnp.max(jnp.diag(gram))):.3e}]"
    )
    return W, elapsed


## Cell 6 — Stage-III budget runner, evaluation, and checkpoint helpers

In [ ]:
# ============================================================
# Cell 6 — Stage-III budget runner, evaluation, and checkpoint helpers
# ============================================================


def coefficient_metrics(ops, params, data_bundle):
    tx = data_bundle["eval_tx"]
    pred_f, pred_sigma = ops["predict_batch"](params, tx[:, 0], tx[:, 1:])
    drift_mse = jnp.mean((pred_f - data_bundle["eval_true_f"]) ** 2)
    diffusion_mse = jnp.mean((pred_sigma - data_bundle["eval_true_sigma"]) ** 2)
    block_until_ready((drift_mse, diffusion_mse))
    return float(drift_mse), float(diffusion_mse)


def checkpoint_path(sde_name, seed):
    safe_name = sde_name.replace("/", "_")
    return CHECKPOINT_DIR / f"{safe_name}__seed_{int(seed)}__{E9_RUN_SIGNATURE}.pkl"


def load_or_train_stage12(problem, ops, data_bundle, seed):
    path = checkpoint_path(problem["name"], seed)
    if CONFIG["E9_RESUME"] and path.exists():
        payload = pickle_load(path)
        if payload.get("run_signature") == E9_RUN_SIGNATURE:
            print(f"[resume] loading Stage-I/II checkpoint: {path}")
            return (
                tree_to_device(payload["baseline_params"]),
                jnp.asarray(payload["W"]),
                float(payload.get("stage1_train_s", np.nan)),
                float(payload.get("stage2_train_s", np.nan)),
            )

    label = f"{problem['name']} seed={seed}"
    baseline_params, stage1_s = train_stage1(
        ops,
        data_bundle["train_data"],
        data_bundle["test_data"],
        data_bundle["root_key"],
        label,
    )
    W, stage2_s = train_stage2(
        ops,
        baseline_params,
        data_bundle["train_data"],
        data_bundle["root_key"],
        label,
    )
    atomic_pickle_dump({
        "run_signature": E9_RUN_SIGNATURE,
        "sde_name": problem["name"],
        "seed": int(seed),
        "baseline_params": tree_to_host(baseline_params),
        "W": np.asarray(jax.device_get(W)),
        "stage1_train_s": float(stage1_s),
        "stage2_train_s": float(stage2_s),
        "n_basis": int(ops["basis"]["n_basis"]),
    }, path)
    return baseline_params, W, stage1_s, stage2_s


def make_fresh_stage3_step(ops):
    # A fresh jitted wrapper makes the separately reported warm-up/compile time
    # meaningful for each budget run.
    @jax.jit
    def stage3_step(params, opt_state, W_fixed, batch, tx_extra):
        return ops["stage3_step_impl"](params, opt_state, W_fixed, batch, tx_extra)
    return stage3_step


def run_stage3_budget(ops, baseline_params, W, train_data, root_key, extra_points, label):
    extra_points = int(extra_points)
    max_extra = int(max(E9_EXTRA_POINT_BUDGETS))
    key_batches = jax.random.fold_in(root_key, 301)
    key_clouds = jax.random.fold_in(root_key, 302)

    # Every budget receives exactly the same minibatch and the same prefix of a
    # common 1024-point jitter cloud at each update.
    def step_inputs(step):
        batch = sample_batch(
            jax.random.fold_in(key_batches, int(step)),
            train_data,
            CONFIG["STAGE3_BATCH_SIZE"],
        )
        if extra_points == 0:
            tx_extra = jnp.zeros((0, Q), dtype=jnp.float32)
        else:
            cloud = sample_tx_data_jitter(
                jax.random.fold_in(key_clouds, int(step)),
                train_data,
                max_extra,
                CONFIG["E9_JITTER_STD"],
            )
            tx_extra = cloud[:extra_points]
        return batch, tx_extra

    gc.collect()
    monitor = PeakMemoryMonitor(CONFIG["MEMORY_POLL_INTERVAL_S"]).start()
    stage3_step = make_fresh_stage3_step(ops)

    # Warm-up/compilation is timed separately and discarded without changing
    # the actual Stage-III starting checkpoint.
    warm_batch, warm_tx = step_inputs(0)
    warm_state = adam_init(baseline_params)
    compile_start = time.perf_counter()
    warm_params, warm_state_out, warm_aux = stage3_step(
        baseline_params, warm_state, W, warm_batch, warm_tx
    )
    block_until_ready((warm_params, warm_state_out, warm_aux))
    compile_s = time.perf_counter() - compile_start
    del warm_params, warm_state_out, warm_aux, warm_state, warm_batch, warm_tx

    params = baseline_params
    opt_state = adam_init(params)
    last_aux = None
    print(
        f"[{label} | S_extra={extra_points:4d}] timed optimization starting "
        f"({CONFIG['STAGE3_UPDATES']} updates; no in-loop logging so timing remains clean)"
    )
    train_start = time.perf_counter()
    for step in range(1, CONFIG["STAGE3_UPDATES"] + 1):
        batch, tx_extra = step_inputs(step)
        params, opt_state, last_aux = stage3_step(
            params, opt_state, W, batch, tx_extra
        )

    block_until_ready((params, opt_state, last_aux))
    train_s = time.perf_counter() - train_start
    memory = monitor.stop()
    total_s = compile_s + train_s
    print(
        f"[{label} | S_extra={extra_points:4d}] done | compile {compile_s:.2f}s | "
        f"train {train_s:.2f}s | total {total_s:.2f}s | peak {memory['peak_mb']:.1f} MB"
    )
    return params, {
        "stage3_compile_s": float(compile_s),
        "stage3_train_s": float(train_s),
        "stage3_total_s": float(total_s),
        "stage3_peak_memory_mb": float(memory["peak_mb"]),
        "stage3_peak_memory_delta_mb": float(memory["delta_mb"]),
        "stage3_memory_source": memory["source"],
    }


def results_pickle_path():
    return OUT_DIR / f"e9_results__{E9_RUN_SIGNATURE}.pkl"


def load_existing_results():
    path = results_pickle_path()
    if CONFIG["E9_RESUME"] and path.exists():
        payload = pickle_load(path)
        if payload.get("run_signature") == E9_RUN_SIGNATURE:
            records = list(payload.get("records", []))
            print(f"[resume] loaded {len(records)} completed Stage-III records from {path}")
            return records
    return []


def save_results(records):
    atomic_pickle_dump({
        "run_signature": E9_RUN_SIGNATURE,
        "records": records,
    }, results_pickle_path())


## Cell 7 — Complete E9 experiment driver

In [ ]:
# ============================================================
# Cell 7 — Complete E9 experiment driver
#
# This is the long-running cell. It produces E9_RESULTS for Cell 8.
# ============================================================

E9_RESULTS = load_existing_results()
completed = {
    (r["sde_name"], int(r["seed"]), int(r["extra_points"]))
    for r in E9_RESULTS
    if r.get("run_signature") == E9_RUN_SIGNATURE
}

print("\n" + "=" * 110)
print("E9: COLLOCATION BUDGET VERSUS ACCURACY, WALL-CLOCK COST, AND PEAK MEMORY")
print("=" * 110)
print("Systems       :", E9_SYSTEMS)
print("Seeds         :", E9_SEEDS)
print("Budgets       :", E9_EXTRA_POINT_BUDGETS)
print("Trajectories  :", CONFIG["N_TRAJECTORIES"])
print("Stage-III runs:", len(E9_SYSTEMS) * len(E9_SEEDS) * len(E9_EXTRA_POINT_BUDGETS))
print("Resume        :", CONFIG["E9_RESUME"])
print("Output dir    :", OUT_DIR)
print("=" * 110)

for system_index, sde_name in enumerate(E9_SYSTEMS):
    problem = make_problem(sde_name)
    ops = build_symmetry_and_training_ops(problem)
    print("\n" + "#" * 110)
    print(f"SYSTEM {system_index + 1}/{len(E9_SYSTEMS)}: {sde_name}")
    print(problem["description"])
    print(f"T={problem['t_final']} | dt={problem['dt']} | x_box={problem['x_box']}")
    print(f"Basis size R={ops['basis']['n_basis']} | breakdown={ops['basis']['counts']}")
    print("#" * 110)

    for seed in E9_SEEDS:
        label = f"{sde_name} seed={seed}"
        needed = [
            b for b in E9_EXTRA_POINT_BUDGETS
            if (sde_name, int(seed), int(b)) not in completed
        ]
        if not needed:
            print(f"[resume] all budgets already complete for {label}")
            continue

        data_bundle = generate_data_bundle(problem, seed, system_index)
        print(
            f"\n[{label}] train transitions={data_bundle['train_data'][0].shape[0]} | "
            f"test transitions={data_bundle['test_data'][0].shape[0]}"
        )

        baseline_params, W, stage1_s, stage2_s = load_or_train_stage12(
            problem, ops, data_bundle, seed
        )
        baseline_drift_mse, baseline_diffusion_mse = coefficient_metrics(
            ops, baseline_params, data_bundle
        )
        baseline_test_nll = float(
            ops["data_nll_loss"](baseline_params, data_bundle["test_data"])
        )
        print(
            f"[{label}] fixed Stage-I metrics | drift MSE {baseline_drift_mse:.6e} | "
            f"diffusion MSE {baseline_diffusion_mse:.6e} | test NLL {baseline_test_nll:.6e}"
        )

        for extra_points in E9_EXTRA_POINT_BUDGETS:
            key = (sde_name, int(seed), int(extra_points))
            if key in completed:
                print(f"[resume] skipping completed {key}")
                continue

            refined_params, timing = run_stage3_budget(
                ops,
                baseline_params,
                W,
                data_bundle["train_data"],
                data_bundle["root_key"],
                extra_points,
                label,
            )
            refined_drift_mse, refined_diffusion_mse = coefficient_metrics(
                ops, refined_params, data_bundle
            )
            refined_test_nll = float(
                ops["data_nll_loss"](refined_params, data_bundle["test_data"])
            )

            record = {
                "run_signature": E9_RUN_SIGNATURE,
                "sde_name": sde_name,
                "seed": int(seed),
                "extra_points": int(extra_points),
                "n_trajectories": int(CONFIG["N_TRAJECTORIES"]),
                "stage3_updates": int(CONFIG["STAGE3_UPDATES"]),
                "n_generators": int(CONFIG["N_GENERATORS"]),
                "n_basis": int(ops["basis"]["n_basis"]),
                "jitter_std": float(CONFIG["E9_JITTER_STD"]),
                "stage1_train_s": float(stage1_s),
                "stage2_train_s": float(stage2_s),
                "baseline_drift_mse": float(baseline_drift_mse),
                "baseline_diffusion_mse": float(baseline_diffusion_mse),
                "baseline_test_nll": float(baseline_test_nll),
                "refined_drift_mse": float(refined_drift_mse),
                "refined_diffusion_mse": float(refined_diffusion_mse),
                "refined_test_nll": float(refined_test_nll),
                **timing,
            }
            E9_RESULTS.append(record)
            completed.add(key)
            save_results(E9_RESULTS)

            drift_red = 100.0 * (baseline_drift_mse - refined_drift_mse) / (
                abs(baseline_drift_mse) + 1e-12
            )
            diff_red = 100.0 * (baseline_diffusion_mse - refined_diffusion_mse) / (
                abs(baseline_diffusion_mse) + 1e-12
            )
            print(
                f"[{label} | S_extra={extra_points:4d}] RESULT | "
                f"drift {refined_drift_mse:.6e} ({drift_red:+.2f}%) | "
                f"diffusion {refined_diffusion_mse:.6e} ({diff_red:+.2f}%) | "
                f"test NLL {refined_test_nll:.6e}"
            )

            del refined_params
            gc.collect()

        del baseline_params, W, data_bundle
        gc.collect()

    del ops
    gc.collect()

print("\nE9 driver complete. Number of records:", len(E9_RESULTS))
print("Saved incremental result file:", results_pickle_path())

## Cell 8 — Final E9 collocation-budget accuracy/compute report

In [ ]:
# ============================================================
# Cell 8 — Final E9 collocation-budget accuracy/compute report
# ADDED final cell. Run only after Cell 7 completes all 75 records.
# ============================================================

from pathlib import Path
import numpy as np
import pandas as pd

if "E9_RESULTS" not in globals():
    raise RuntimeError("Run the E9 experiment-driver cell first; E9_RESULTS is missing.")

raw = pd.DataFrame(E9_RESULTS).copy()
required = [
    "sde_name", "seed", "extra_points", "n_trajectories", "stage3_updates",
    "baseline_drift_mse", "baseline_diffusion_mse", "baseline_test_nll",
    "refined_drift_mse", "refined_diffusion_mse", "refined_test_nll",
    "stage3_compile_s", "stage3_train_s", "stage3_total_s",
    "stage3_peak_memory_mb", "stage3_peak_memory_delta_mb",
]
missing = [c for c in required if c not in raw.columns]
if missing:
    raise RuntimeError(f"E9_RESULTS is missing fields: {missing}")

systems = list(globals().get("E9_SYSTEMS", [
    "ring_double_well_50d", "low_rank_ou_50d", "tanh_chain_50d"
]))
budgets = [int(x) for x in globals().get(
    "E9_EXTRA_POINT_BUDGETS", [0, 128, 256, 512, 1024]
)]
seeds = [int(x) for x in globals().get("E9_SEEDS", [0, 1, 2, 3, 4])]
out_dir = Path(globals().get("E9_OUTPUT_DIR", "./e9_outputs"))
eps = 1e-12

for c in ["seed", "extra_points", "n_trajectories", "stage3_updates"]:
    raw[c] = raw[c].astype(int)
raw = raw.sort_values(["sde_name", "seed", "extra_points"]).reset_index(drop=True)

# The report refuses incomplete or unfair sweeps.
expected = pd.MultiIndex.from_product(
    [systems, seeds, budgets], names=["sde_name", "seed", "extra_points"]
)
observed = pd.MultiIndex.from_frame(raw[["sde_name", "seed", "extra_points"]])
if raw.duplicated(["sde_name", "seed", "extra_points"]).any():
    raise RuntimeError("Duplicate (sde_name, seed, extra_points) records found.")
if len(expected.difference(observed)) or len(observed.difference(expected)):
    raise RuntimeError(
        "The E9 Cartesian sweep is incomplete or contains unexpected runs.\n"
        f"Missing: {list(expected.difference(observed))[:20]}\n"
        f"Unexpected: {list(observed.difference(expected))[:20]}"
    )
if not (raw["n_trajectories"] == 60).all():
    raise RuntimeError("E9 requires the data-scarce 50D setting N_trajectories=60.")
if raw["stage3_updates"].nunique() != 1:
    raise RuntimeError("All budgets must use the same Stage-III update count.")

numeric = [c for c in required if c not in {
    "sde_name", "seed", "extra_points", "n_trajectories", "stage3_updates"
}]
if not np.isfinite(raw[numeric].to_numpy(float)).all():
    raise RuntimeError("E9_RESULTS contains NaN or infinite values.")

# Verify that each budget starts from the same Stage-I checkpoint.
for metric in ["baseline_drift_mse", "baseline_diffusion_mse", "baseline_test_nll"]:
    spread = raw.groupby(["sde_name", "seed"])[metric].agg(lambda x: x.max() - x.min())
    scale = raw.groupby(["sde_name", "seed"])[metric].agg(
        lambda x: max(abs(float(x.mean())), 1.0)
    )
    if ((spread / scale) > 1e-9).any():
        raise RuntimeError(f"{metric} changes across budgets for a fixed system/seed.")

# Primary E9 quantities.
raw["stage3_per_step_ms"] = 1000.0 * raw.stage3_train_s / raw.stage3_updates
raw["drift_reduction_vs_baseline_pct"] = 100.0 * (
    raw.baseline_drift_mse - raw.refined_drift_mse
) / np.maximum(np.abs(raw.baseline_drift_mse), eps)
raw["diffusion_reduction_vs_baseline_pct"] = 100.0 * (
    raw.baseline_diffusion_mse - raw.refined_diffusion_mse
) / np.maximum(np.abs(raw.baseline_diffusion_mse), eps)
raw["test_nll_change"] = raw.refined_test_nll - raw.baseline_test_nll

# Pair each run with S_extra=0 from the same system and seed. This isolates
# the value of extra collocation from the on-trajectory symmetry refinement.
s0 = raw.loc[raw.extra_points == 0, [
    "sde_name", "seed", "refined_drift_mse", "refined_diffusion_mse", "stage3_total_s"
]].rename(columns={
    "refined_drift_mse": "s0_drift_mse",
    "refined_diffusion_mse": "s0_diffusion_mse",
    "stage3_total_s": "s0_total_s",
})
raw = raw.merge(s0, on=["sde_name", "seed"], how="left", validate="many_to_one")
raw["drift_gain_vs_s0_pct"] = 100.0 * (
    raw.s0_drift_mse - raw.refined_drift_mse
) / np.maximum(np.abs(raw.s0_drift_mse), eps)
raw["diffusion_gain_vs_s0_pct"] = 100.0 * (
    raw.s0_diffusion_mse - raw.refined_diffusion_mse
) / np.maximum(np.abs(raw.s0_diffusion_mse), eps)
raw["joint_gain_vs_s0_pp"] = 0.5 * (
    raw.drift_gain_vs_s0_pct + raw.diffusion_gain_vs_s0_pct
)
raw["added_total_s_vs_s0"] = raw.stage3_total_s - raw.s0_total_s
raw["runtime_ratio_vs_s0"] = raw.stage3_total_s / np.maximum(raw.s0_total_s, eps)
raw["joint_gain_pp_per_added_s"] = np.where(
    raw.added_total_s_vs_s0 > 0,
    raw.joint_gain_vs_s0_pp / raw.added_total_s_vs_s0,
    np.nan,
)
raw["both_coefficients_beat_s0"] = (
    (raw.drift_gain_vs_s0_pct > 0) & (raw.diffusion_gain_vs_s0_pct > 0)
).astype(float)

# Mean and seed standard deviation for every system/budget.
metrics = [
    "baseline_drift_mse", "refined_drift_mse", "drift_reduction_vs_baseline_pct",
    "baseline_diffusion_mse", "refined_diffusion_mse", "diffusion_reduction_vs_baseline_pct",
    "baseline_test_nll", "refined_test_nll", "test_nll_change",
    "stage3_compile_s", "stage3_train_s", "stage3_total_s", "stage3_per_step_ms",
    "stage3_peak_memory_mb", "stage3_peak_memory_delta_mb",
    "added_total_s_vs_s0", "runtime_ratio_vs_s0",
    "drift_gain_vs_s0_pct", "diffusion_gain_vs_s0_pct", "joint_gain_vs_s0_pp",
    "joint_gain_pp_per_added_s", "both_coefficients_beat_s0",
]
summary = raw.groupby(["sde_name", "extra_points"])[metrics].agg(["mean", "std"])
summary.columns = [f"{m}_{s}" for m, s in summary.columns]
summary = summary.reset_index()
summary["n_seeds"] = len(seeds)
summary["both_coefficients_beat_s0_pct"] = 100.0 * summary.both_coefficients_beat_s0_mean

# System-balanced overall result: average the three systems within each seed,
# then report mean/std across paired seeds.
seed_balanced = raw.groupby(["seed", "extra_points"], as_index=False).agg(
    drift_reduction_pct=("drift_reduction_vs_baseline_pct", "mean"),
    diffusion_reduction_pct=("diffusion_reduction_vs_baseline_pct", "mean"),
    drift_gain_vs_s0_pct=("drift_gain_vs_s0_pct", "mean"),
    diffusion_gain_vs_s0_pct=("diffusion_gain_vs_s0_pct", "mean"),
    joint_gain_vs_s0_pp=("joint_gain_vs_s0_pp", "mean"),
    total_s=("stage3_total_s", "mean"),
    per_step_ms=("stage3_per_step_ms", "mean"),
    peak_memory_mb=("stage3_peak_memory_mb", "mean"),
    peak_memory_delta_mb=("stage3_peak_memory_delta_mb", "mean"),
    added_s=("added_total_s_vs_s0", "mean"),
    runtime_ratio=("runtime_ratio_vs_s0", "mean"),
    gain_pp_per_added_s=("joint_gain_pp_per_added_s", "mean"),
    both_win_rate=("both_coefficients_beat_s0", "mean"),
)
overall = seed_balanced.groupby("extra_points").agg(["mean", "std"])
overall.columns = [f"{m}_{s}" for m, s in overall.columns]
overall = overall.reset_index().sort_values("extra_points")
overall["both_win_rate_mean"] *= 100.0
overall["both_win_rate_std"] *= 100.0
overall["marginal_gain_pp_per_128_points"] = (
    overall.joint_gain_vs_s0_pp_mean.diff() / overall.extra_points.diff() * 128.0
)
overall["marginal_gain_pp_per_added_s"] = np.where(
    overall.total_s_mean.diff() > 0,
    overall.joint_gain_vs_s0_pp_mean.diff() / overall.total_s_mean.diff(),
    np.nan,
)

# Numerical saturation summary.
system_means = raw.groupby(["sde_name", "extra_points"], as_index=False).agg(
    drift_mse=("refined_drift_mse", "mean"),
    diffusion_mse=("refined_diffusion_mse", "mean"),
    joint_gain_pp=("joint_gain_vs_s0_pp", "mean"),
    efficiency=("joint_gain_pp_per_added_s", "mean"),
)
sat_rows = []
for name, g in system_means.groupby("sde_name"):
    g = g.sort_values("extra_points")
    best = max(float(g.joint_gain_pp.max()), 0.0)
    capture = 100.0 * g.joint_gain_pp / best if best > eps else np.full(len(g), np.nan)
    eligible = g.loc[np.asarray(capture) >= 90.0]
    eff = g.loc[(g.extra_points > 0) & np.isfinite(g.efficiency)]
    sat_rows.append({
        "sde_name": name,
        "smallest_budget_at_90pct_best_joint_gain": (
            int(eligible.iloc[0].extra_points) if len(eligible) else np.nan
        ),
        "best_joint_gain_budget": int(g.loc[g.joint_gain_pp.idxmax(), "extra_points"]),
        "best_drift_budget": int(g.loc[g.drift_mse.idxmin(), "extra_points"]),
        "best_diffusion_budget": int(g.loc[g.diffusion_mse.idxmin(), "extra_points"]),
        "best_efficiency_budget": (
            int(eff.loc[eff.efficiency.idxmax(), "extra_points"]) if len(eff) else np.nan
        ),
    })
saturation = pd.DataFrame(sat_rows)

# Save all report-ready numbers.
out_dir.mkdir(parents=True, exist_ok=True)
files = {
    "raw": out_dir / "e9_raw_per_seed.csv",
    "system_budget": out_dir / "e9_summary_by_system_budget.csv",
    "overall": out_dir / "e9_overall_budget_summary.csv",
    "saturation": out_dir / "e9_saturation_summary.csv",
}
raw.to_csv(files["raw"], index=False)
summary.to_csv(files["system_budget"], index=False)
overall.to_csv(files["overall"], index=False)
saturation.to_csv(files["saturation"], index=False)

pd.set_option("display.width", 260)
pd.set_option("display.max_columns", 200)
pd.set_option("display.max_rows", 500)
fmt = lambda x: f"{x:.6e}"

print("\n" + "=" * 118)
print("FINAL E9 REPORT — COLLOCATION BUDGET VS ACCURACY, TIME, AND MEMORY")
print("=" * 118)
print(f"Systems                  : {systems}")
print(f"Extra-point budgets      : {budgets}")
print(f"Seeds                    : {seeds}")
print(f"N trajectories           : {int(raw.n_trajectories.iloc[0])}")
print(f"Stage-III updates/run    : {int(raw.stage3_updates.iloc[0])}")
print("S_extra=0 meaning       : on-trajectory symmetry points only; no extra jittered points")
print("Total-time definition   : compile/warm-up + timed Stage-III optimization")
print("Efficiency definition   : mean drift/diffusion gain over S_extra=0 per added second")
try:
    import jax
    print(f"Backend/device           : {jax.default_backend()} / {jax.devices()[0]}")
except Exception:
    pass

counts = raw.pivot_table(
    index="sde_name", columns="extra_points", values="seed", aggfunc="count"
).reindex(index=systems, columns=budgets)
print("\n[1] COMPLETENESS: number of seeds")
print(counts.to_string())

acc_cols = [
    "sde_name", "extra_points", "n_seeds",
    "baseline_drift_mse_mean", "refined_drift_mse_mean", "refined_drift_mse_std",
    "drift_reduction_vs_baseline_pct_mean", "drift_reduction_vs_baseline_pct_std",
    "baseline_diffusion_mse_mean", "refined_diffusion_mse_mean", "refined_diffusion_mse_std",
    "diffusion_reduction_vs_baseline_pct_mean", "diffusion_reduction_vs_baseline_pct_std",
    "test_nll_change_mean", "test_nll_change_std",
]
print("\n[2] ACCURACY BY SYSTEM/BUDGET: mean and seed standard deviation")
print(summary[acc_cols].to_string(index=False, float_format=fmt))

cost_cols = [
    "sde_name", "extra_points", "n_seeds",
    "stage3_total_s_mean", "stage3_total_s_std",
    "stage3_per_step_ms_mean", "stage3_per_step_ms_std",
    "stage3_peak_memory_mb_mean", "stage3_peak_memory_mb_std",
    "stage3_peak_memory_delta_mb_mean", "stage3_peak_memory_delta_mb_std",
    "added_total_s_vs_s0_mean", "runtime_ratio_vs_s0_mean",
    "drift_gain_vs_s0_pct_mean", "diffusion_gain_vs_s0_pct_mean",
    "joint_gain_vs_s0_pp_mean", "joint_gain_vs_s0_pp_std",
    "joint_gain_pp_per_added_s_mean", "joint_gain_pp_per_added_s_std",
    "both_coefficients_beat_s0_pct",
]
print("\n[3] ACCURACY/COST TRADEOFF BY SYSTEM/BUDGET")
print(summary[cost_cols].to_string(index=False, float_format=fmt))

print("\n[4] SYSTEM-BALANCED OVERALL SUMMARY: mean and std across paired seeds")
print(overall.to_string(index=False, float_format=fmt))

print("\n[5] SATURATION SUMMARY")
print(saturation.to_string(index=False))

all_sat = saturation.smallest_budget_at_90pct_best_joint_gain.notna().all()
all_sat = bool(all_sat and (saturation.smallest_budget_at_90pct_best_joint_gain <= 512).all())
both_positive = False
for b in [256, 512]:
    q = summary.loc[summary.extra_points == b]
    both_positive |= bool(((
        q.drift_gain_vs_s0_pct_mean > 0
    ) & (
        q.diffusion_gain_vs_s0_pct_mean > 0
    )).all())
print("\n[6] CONSERVATIVE OPERATIONAL INTERPRETATION")
print(f"All systems reach >=90% of best joint gain by 512 points : {all_sat}")
print(f"At 256 or 512, both mean coefficient errors beat S0      : {both_positive}")
if all_sat and both_positive:
    print("Evidence: consistent with useful saturation around 256–512 extra points.")
elif both_positive:
    print("Evidence: favorable Pareto tradeoff, but saturation is not uniform.")
else:
    print("Evidence: no uniform two-coefficient benefit from extra collocation yet.")

raw_cols = [
    "sde_name", "seed", "extra_points",
    "baseline_drift_mse", "refined_drift_mse", "drift_reduction_vs_baseline_pct",
    "baseline_diffusion_mse", "refined_diffusion_mse", "diffusion_reduction_vs_baseline_pct",
    "baseline_test_nll", "refined_test_nll", "test_nll_change",
    "stage3_compile_s", "stage3_train_s", "stage3_total_s", "stage3_per_step_ms",
    "stage3_peak_memory_mb", "stage3_peak_memory_delta_mb",
    "added_total_s_vs_s0", "runtime_ratio_vs_s0",
    "drift_gain_vs_s0_pct", "diffusion_gain_vs_s0_pct",
    "joint_gain_vs_s0_pp", "joint_gain_pp_per_added_s",
]
print("\n[7] RAW PER-SEED NUMBERS")
print(raw[raw_cols].to_string(index=False, float_format=lambda x: f"{x:.8e}"))

print("\n[8] SAVED CSV FILES")
for label, path in files.items():
    print(f"{label:14s}: {path}")
print("=" * 118)
